# Lección 2B · Comunicación serial, muestreo y registro de datos

**Tópicos de Ingeniería Eléctrica II · Semana 2 · UACJ**

**Propósito:** preparar la adquisición experimental por USB que se implementará en la Semana 3. En este cuaderno se utilizan **datos simulados**, sin ESP32.

**Resultados de aprendizaje:** identificar cada etapa del flujo de datos; interpretar un registro CSV; aplicar NumPy y Matplotlib; y distinguir *muestreo*, *recepción*, *registro* y *visualización*.

**Secuencia:** fundamento → código ejecutable → predicción → reto → comprobación. Cada estudiante debe documentar sus conclusiones y reconocer qué datos son simulados.

## 1. Arquitectura de adquisición

**Sensor/entrada → ADC → ESP32 → USB serial → PySerial → tratamiento NumPy → gráfica Matplotlib → archivo CSV.**

El ESP32 toma muestras a un periodo aproximadamente fijo. La computadora **recibe** las tramas y las almacena; el refresco de la gráfica puede ser más lento que el muestreo sin que se pierdan necesariamente los registros.

Una trama del experimento será:

```text
t_ms,adc,voltage
100,2048,1.6504
200,2132,1.7181
```

Aquí `voltage` es una **estimación** obtenida con la escala nominal de 3.3 V y 12 bits, no una medición calibrada.

## 2. Intervalo de muestreo y resolución

Para un muestreo uniforme, se definen el período \(T_s\), la frecuencia \(f_s\), y el número de niveles de un ADC ideal de \(N\) bits:

\[
f_s=\frac{1}{T_s},\qquad \Delta V_{\mathrm{ideal}}=\frac{V_{\mathrm{FS}}}{2^N-1}.
\]

**Nota:** la respuesta y la calibración reales del ADC del ESP32 deben comprobarse experimentalmente.

In [ ]:
import numpy as np
fs = 10.0   # muestras por segundo
Ts = 1.0/fs
N = 12
VFS = 3.3
print(f"Período: {Ts*1000:.0f} ms | Resolución ideal: {VFS/(2**N-1)*1000:.3f} mV/cuenta")

**Reto 2B.1:** modifique `fs` a 5 y 20 Hz. Explique por qué esto cambia el tiempo entre muestras, pero no la resolución ideal del ADC.

## 3. Protocolo serial y validación de tramas

La comunicación se define antes de programar la interfaz. Cada línea contiene tres campos numéricos separados por comas. Se descartan encabezados, mensajes de diagnóstico y líneas incompletas.

In [ ]:
import math

def parse_frame(linea):
    s = linea.strip()
    if not s or s.startswith('#') or s.lower() == 't_ms,adc,voltage':
        return None
    partes = s.split(',')
    if len(partes) != 3:
        return None
    try:
        t_ms = int(partes[0]); adc = int(partes[1]); v = float(partes[2])
    except ValueError:
        return None
    if t_ms < 0 or not 0 <= adc <= 4095 or not math.isfinite(v) or not -0.1 <= v <= 3.5:
        return None
    return t_ms, adc, v

tramas = ['t_ms,adc,voltage', '100,2048,1.6504', '200,2132,1.7181', '300,ERROR,1.72', '400,2300']
validas = [registro for linea in tramas if (registro := parse_frame(linea)) is not None]
print('Registros válidos:', validas)
print('Descartados:', len(tramas)-1-len(validas))

**Reto 2B.2:** agregue dos líneas incorrectas y explique qué validación impidió incorporarlas al registro.

## 4. Emulador de muestreo a 10 Hz

Se genera una señal lenta (0.4 Hz) y se codifica como lo haría el emisor serial del ejercicio. Se utiliza un generador reproducible. **No hay lectura del puerto USB en este bloque.**

In [ ]:
import matplotlib.pyplot as plt
fs = 10.0
segundos = 12.0
t = np.arange(0, segundos, 1/fs)
voltaje_ideal = 1.65 + 0.85*np.sin(2*np.pi*0.4*t)
cuentas = np.clip(np.rint(voltaje_ideal/3.3*4095),0,4095).astype(int)
lineas = [f"{int(round(ti*1000))},{int(a)},{(3.3*a/4095):.4f}" for ti,a in zip(t,cuentas)]
registros = [parse_frame(s) for s in lineas]
print('Tramas generadas:',len(lineas),'| Primeras 3:',lineas[:3])
assert all(x is not None for x in registros)

In [ ]:
tm,adc,v = np.array(registros).T
fig,ax=plt.subplots(figsize=(9,3.5))
ax.plot(tm/1000,v,lw=1.8,label='Voltaje reconstruido')
ax.set(xlabel='Tiempo (s)',ylabel='Voltaje (V)',title='Señal de prueba · datos simulados')
ax.grid(True,alpha=.25); ax.legend(); plt.show()

**Reto 2B.3:** cambie la frecuencia de la señal de 0.4 Hz a 2 Hz y luego a 7 Hz. Analice qué ocurre al mantener \(f_s=10\) Hz. Use el criterio de Nyquist como referencia teórica, sin confundirlo con la calibración del ADC.

## 5. Registro y lectura de CSV

Se almacenan las muestras en columnas con unidades explícitas. El archivo se puede descargar desde Google Colab mediante su explorador de archivos.

In [ ]:
import csv
nombre = 'semana02B_adquisicion_simulada.csv'
with open(nombre,'w',newline='',encoding='utf-8') as f:
    escritor = csv.writer(f)
    escritor.writerow(['t_ms','adc','voltage'])
    escritor.writerows(registros)
with open(nombre,newline='',encoding='utf-8') as f:
    filas = list(csv.DictReader(f))
print('CSV:',nombre,'| filas:',len(filas),'| inicio:',filas[0])

## 6. PySerial: conexión real frente a emulación

En una computadora con Python local se puede abrir un puerto con:

```python
import serial
ser = serial.Serial('COM5', baudrate=115200, timeout=1)
try:
    linea = ser.readline().decode('utf-8', errors='replace')
    print(linea)
finally:
    ser.close()
```

En Linux el nombre puede ser `/dev/ttyUSB0` o `/dev/ttyACM0`. **Estos ejemplos NO deben ejecutarse con un nombre de puerto supuesto**: hay que comprobar el puerto real.

**Importante:** Google Colab se ejecuta en un servidor remoto y normalmente **no tiene acceso directo al puerto USB** de la computadora del estudiante. Para la práctica real utilizaremos una aplicación Python instalada localmente (Semana 3).

La opción de abajo muestra un *loopback virtual* de PySerial, que prueba el intercambio lógico de texto sin ESP32. Si PySerial no está instalado, ejecute primero `%pip install pyserial` en una celda independiente.

In [ ]:
try:
    import serial
except ImportError:
    serial = None
    print('PySerial no está instalado. Ejecute en Colab una celda: %pip install pyserial')

if serial is not None:
    with serial.serial_for_url('loop://', timeout=1) as enlace:
        enlace.write(b'100,2048,1.6504\n')
        recibido = enlace.readline().decode('ascii').strip()
        print('Trama virtual:', recibido)
        print('Registro:', parse_frame(recibido))

## 7. Programación basada en eventos

Una aplicación de supervisión no debe congelarse esperando una lectura. En una interfaz Tkinter, una función puede revisar el puerto sin bloquear y programar la siguiente actualización mediante `root.after(100, actualizar)`.

La adquisición real de la Semana 3 aplicará este principio, con una ventana y gráficas actualizadas periódicamente. El ESP32 será responsable del instante de muestreo; la computadora, del registro y de la visualización.

## 8. Verificación y actividad con IA

1. ¿Qué diferencia existe entre un período de muestreo de 100 ms y una gráfica que se refresca cada 200 ms?
2. ¿Por qué debemos registrar la marca de tiempo transmitida por el microcontrolador?
3. ¿Qué sucede con una trama que contiene `abc` en lugar de una cuenta ADC?
4. ¿Qué información necesita para saber si una tensión es calibrada o solo estimada?
5. Entregue el `.ipynb` ejecutado, una gráfica, un CSV y una explicación del flujo de datos.

**Tutor de IA (actividad guiada):** «Actúe como tutor socrático de adquisición de datos. Hágame una pregunta por vez sobre muestreo, PySerial, validación de tramas y registro CSV. Revise mis respuestas, sin darme inmediatamente el programa terminado». Documente lo que consultó y cómo comprobó cada respuesta.

**Puente hacia Semana 3:** subir el firmware al ESP32, recibir sus líneas por USB y registrar las muestras desde la aplicación local.